# 07 · RAG 100% local: Weaviate + Ollama

Embeddings **e** LLM rodando no laptop, sem chave de API. O Weaviate chama o Ollama sozinho, tanto para vetorizar quanto para gerar a resposta.

Pré-requisitos:

```sh
ollama pull bge-m3      # embeddings multilíngues
ollama pull llama3.2    # LLM pequeno
```

O cluster já sobe com `ENABLE_MODULES=text2vec-ollama,generative-ollama`. De dentro do container, o Ollama do host fica em `host.docker.internal:11434`.

In [1]:
import weaviate
from weaviate.classes.config import Configure
import utils

client = weaviate.connect_to_local()
print([m for m in client.get_meta()["modules"] if "ollama" in m])

['generative-ollama', 'text2vec-ollama']


/Users/dudanogueira/dev/weaviate/lab/claude/latinoware-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
wiki = utils.load_wiki(1)
textos = wiki.text.head(300).tolist() + wiki[wiki.text.str.contains("volcan")].text.head(30).tolist()

OLLAMA = "http://host.docker.internal:11434"
client.collections.delete("Docs")
docs = client.collections.create(
    "Docs",
    vector_config=Configure.Vectors.text2vec_ollama(api_endpoint=OLLAMA, model="bge-m3"),
    generative_config=Configure.Generative.ollama(api_endpoint=OLLAMA, model="llama3.2"),
)
resultado = docs.data.insert_many([{"text": t} for t in textos])  # o Weaviate vetoriza com o Ollama
print(len(textos), "textos | erros:", len(resultado.errors))
utils.wait_for_indexing(client, "Docs")  # com ASYNC_INDEXING, a busca só enxerga o que já foi indexado

330 textos | erros: 0


## Pergunta → busca → resposta

In [3]:
r = docs.generate.near_text(
    query="como os vulcões entram em erupção?",
    limit=3,
    grouped_task="Responda em português usando só estes trechos",
)
for o in r.objects:
    print("-", o.properties["text"][:100], "…")
print()
print(r.generative.text)

- Intrusive volcanism is when magma is forced into the rocks that make up the Earth's crust.   When it …
- The main ways that carbon gets into the carbon cycle are volcanoes, and the burning of fossil fuels  …
- In geology, volcanology (also spelled vulcanology) is the part of geology that studies volcanoes, la …

Volcanismo intrusivo ocorre quando magma é empurrado para dentro da crosta terrestre. Quando se resfria e se torna sólida ainda sob a superfície, diferentes características chamadas plutônos são formadas. O tipo de rocha formada é rocha ígena intrusiva.


Para trocar de provedor, basta mudar uma linha: `text2vec_ollama` → `text2vec_huggingface`, `text2vec_openai`, `text2vec_cohere`, `text2vec_weaviate`…

In [4]:
client.close()